In [ ]:
# ============================================================
# LADDER — 4-Method Ablation (Tie-Aware)
# ============================================================
library(ggplot2)
library(dplyr)
library(tidyr)
library(patchwork)
library(ggdist)
library(scales)

# === FILE PATH ===

csv_path  <- "ladder_validation_outputs/Ablation_validation_results_general_only.csv"
run_label <- "AML — Annotation Stage Comparison"

keep_models <- c("BioLORD-2023")

method_keys <- c("Enrichment", "Direct", "Final_no_val", "Final_val")

method_labels <- c(
  Enrichment   = "Enrichment-driven LLM\n(no validation)",
  Direct       = "Direct LLM\n(no validation)",
  Final_no_val = "Final annotation\n(no validation)",
  Final_val    = "Final annotation\n(with validation)"
)

method_pal <- c(
  Enrichment   = "#f4a582",
  Direct       = "#0571b0",
  Final_no_val = "#ca0020",
  Final_val    = "#4dac26"
)

theme_nature <- function(base_size = 10) {
  theme_classic(base_size = base_size, base_family = "Helvetica") +
  theme(
    panel.border       = element_rect(colour = "black", fill = NA, linewidth = 0.6),
    panel.grid.major.y = element_line(colour = "grey92", linewidth = 0.35),
    panel.grid.minor   = element_blank(),
    axis.line          = element_blank(),
    axis.ticks         = element_line(colour = "black", linewidth = 0.45),
    axis.ticks.length  = unit(3, "pt"),
    axis.title         = element_text(face = "bold", size = base_size),
    axis.text          = element_text(colour = "black", size = base_size - 1),
    axis.text.x        = element_text(angle = 0, hjust = 0.5),  # single model: no rotation needed
    legend.title       = element_text(face = "bold", size = base_size - 1),
    legend.text        = element_text(size = base_size - 2),
    legend.key         = element_blank(),
    legend.background  = element_blank(),
    plot.title         = element_text(face = "bold", size = base_size + 1, hjust = 0),
    plot.subtitle      = element_text(size = base_size - 2, colour = "grey45", hjust = 0),
    strip.background   = element_rect(fill = "grey96", colour = "black", linewidth = 0.5),
    strip.text         = element_text(face = "bold", size = base_size - 1),
    plot.margin        = margin(8, 12, 8, 8)
  )
}


raw_all <- read.csv(csv_path, stringsAsFactors = FALSE) %>%
  mutate(CancerType = case_when(
    grepl("BREAST", Geneset, ignore.case = TRUE) ~ "Breast",
    grepl("LUNG",   Geneset, ignore.case = TRUE) ~ "Lung",
    TRUE ~ "AML"
  ))


raw <- raw_all %>%
  filter(CancerType == "AML") %>%
  filter(Model %in% keep_models) %>%
  mutate(Model = factor(Model, levels = keep_models))

sim_long <- raw %>%
  select(Geneset, Model,
         Enrichment   = Enrichment_Similarity,
         Direct       = Direct_Similarity,
         Final_no_val = Final_no_val_Similarity,
         Final_val    = Final_val_Similarity) %>%
  pivot_longer(cols      = all_of(method_keys),
               names_to  = "Method",
               values_to = "Similarity") %>%
  mutate(Method = factor(Method, levels = method_keys))


winners_long <- raw %>%
  select(Geneset, Model, Winner) %>%
  mutate(Winner = strsplit(gsub("\\s*[|,]\\s*", ",", Winner), ",")) %>%
  unnest(Winner) %>%
  mutate(
    Winner = trimws(Winner),
    Method = factor(Winner, levels = method_keys)
  ) %>%
  filter(!is.na(Method)) %>%
  select(Geneset, Model, Method) %>%
  mutate(IsWinner = TRUE)

# Join winner flag back to similarity long
results <- sim_long %>%
  left_join(winners_long, by = c("Geneset", "Model", "Method")) %>%
  mutate(IsWinner = ifelse(is.na(IsWinner), FALSE, IsWinner))

# ============================================================
# PANEL A — Win counts
# ============================================================
wins <- results %>%
  filter(IsWinner) %>%
  count(Model, Method) %>%
  complete(Model, Method, fill = list(n = 0))

pA <- ggplot(wins, aes(x = Model, y = n, fill = Method)) +
  geom_col(position = position_dodge(width = 0.85), width = 0.75,
           colour = "white", linewidth = 0.4) +
  geom_text(aes(label = ifelse(n > 0, n, "")),
            position = position_dodge(width = 0.85),
            vjust = -0.4, size = 2.3, fontface = "bold", colour = "black") +
  scale_fill_manual(values = method_pal, labels = method_labels, drop = FALSE) +
  scale_x_discrete(expand = expansion(add = 0.6)) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.15)),
                     breaks = pretty_breaks(4)) +
  labs(
    title    = run_label,
    subtitle = "Win counts by embedding model (tie-aware)",
    x        = "Embedding Model",
    y        = "Number of Wins",
    fill     = "Annotation Stage"
  ) +
  theme_nature() +
  theme(panel.grid.major.x = element_blank())

# ============================================================
# PANEL B — Cosine similarity raincloud
# ============================================================
pB <- ggplot(results, aes(x = Model, y = Similarity,
                           fill = Method, colour = Method)) +
  stat_halfeye(adjust = 0.8, width = 0.45, .width = 0,
               justification = -0.2, point_colour = NA, alpha = 0.72,
               position = position_dodge(width = 0.85)) +
  geom_boxplot(outlier.shape = NA, width = 0.13, linewidth = 0.45,
               colour = "black", alpha = 0.55,
               position = position_dodge(width = 0.85)) +
  geom_point(position = position_jitterdodge(jitter.width = 0.04,
                                              dodge.width  = 0.85,
                                              seed = 42),
             size = 0.7, alpha = 0.3, shape = 16) +
  scale_fill_manual(values = method_pal, labels = method_labels, drop = FALSE) +
  scale_colour_manual(values = method_pal, labels = method_labels, drop = FALSE) +
  scale_x_discrete(expand = expansion(add = 0.6)) +
  scale_y_continuous(breaks = seq(0, 1, 0.2), limits = c(NA, 1.05)) +
  labs(
    title    = run_label,
    subtitle = "Cosine similarity distribution by annotation stage",
    x        = "Embedding Model",
    y        = "Cosine Similarity",
    fill     = "Annotation Stage"
  ) +
  theme_nature() +
  guides(colour = "none")   # suppress duplicate legend — fill already shows Annotation Stage

# ============================================================
# ASSEMBLE & SAVE
# ============================================================
fig <- (pA | pB) +
  plot_layout(guides = "collect") &
  theme(legend.position  = "bottom",
        legend.direction = "horizontal") &
  guides(fill = guide_legend(nrow = 2, byrow = TRUE))  # wrap legend so it fits width, avoids clipping

ggsave("Fig_LADDER_AML_4MethodsBiolord.pdf", fig, width = 12, height = 7, dpi = 300)
ggsave("Fig_LADDER_AML_4MethodsBiolord.png", fig, width = 12, height = 7, dpi = 300)

cat("Saved Fig_LADDER_AML_4MethodsBiolord.pdf/.png\n")